In [1]:
# coding: utf-8
import numpy as np
import pandas as pd

import random
import math

import seaborn as sns
import matplotlib.pyplot as plt

import tourn as t

# добавить папку с собственными библиотеками
import sys
sys.path.append('/home/roman/python/chgk/google_docs')
import google_drive as gd

# увеличим дефолтный размер графиков
from pylab import rcParams
rcParams['figure.figsize'] = 8, 5

pd.set_option('display.max_rows', 45000)
pd.set_option('display.max_columns', 50000)
pd.set_option('display.max_colwidth', 5000)

/home/roman/.local/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [12]:
n_iterations = 100
teams = 32
real_id = 580027
barca_id = 580023

In [13]:
gd_teams_df = gd.read('1bksSTCuXPyeuni9wBf8O8JJgUfBj-YUUG27KpJEOK80', "clubelo-club-rankings!A2:C5000")
gd_teams_df.columns = ['team_id', 'team_name', 'start_rating']

# proper data and dict
team_df = gd_teams_df
team_df['is_active'] = 1
team_df['main_value'] = np.random.randint(0, 100, size=len(team_df)) / 1000
team_df = team_df.set_index('team_id')
teams_dct = team_df[0:teams].to_dict('index')

In [14]:
classico_flas_lst = []
res_df = pd.DataFrame()
for itr in range(n_iterations):
    # playof - fast and popular way, bit hig dispersional
    # log with game results after simulation
    # half seed - seed by start rating, best 2 teams can meet only in final
    log_df = t.simulate_playoff_tourn(teams_dct, seed_mode='half')
    # usefull table to look at different parameters
    team_df = t.team_table(log_df, teams_dct)
    # tournament table
    po_df = t.make_res_table(log_df, teams_dct)

    # round robin - the best for dispersion, but long schedule
    # two rounds by default
    schedule = t.get_round_robin_schedule(teams_dct)
    # log with game results after simulation
    log_df = t.simulate_matches_by_schedule(teams_dct, schedule, total=(0,1), sigma=(2.5, 2.6), last_match_id=0)
    # tournament table
    liga_df = t.make_res_table(log_df, teams_dct)

    # сначала финалисты
    po_df = po_df.sort_values(by='playoff_place', ascending=True)
    sc_df_teams_lst = list(po_df['team_id'].astype(int).values[0:2])
    # добираем из чемпионата пока число участников не станет 4
    liga_df = liga_df.sort_values(by='roundrobin_place', ascending=True)
    i = 0
    while len(sc_df_teams_lst) < 4:
        cand = liga_df['team_id'].iloc[i]
        if cand not in sc_df_teams_lst:
            sc_df_teams_lst.append(cand)
        i += 1

    sc_dct = {}
    for i in range(len(sc_df_teams_lst)):
        sc_dct[sc_df_teams_lst[i]] = teams_dct[sc_df_teams_lst[i]]



    log_df = t.simulate_playoff_tourn(sc_dct, seed_mode='half')
    # usefull table to look at different parameters
    team_df = t.team_table(log_df, sc_dct)
    # tournament table
    sc_df = t.make_res_table(log_df, sc_dct)
    sc_df['supercup_place'] = sc_df['playoff_place']

    sc_final_teams = sc_df[sc_df['playoff_place'] < 3]['team_id'].values
    sc_final_teams.sort()

    classico_flag = 0
    if (real_id in sc_final_teams) and (barca_id in sc_final_teams):
        classico_flag = 1
    else:
        classico_flag = 0

    classico_flas_lst.append(classico_flag)

    iteration_df = liga_df.copy()[[
        'team_id', 'team_name', 'roundrobin_place'
    ]]
    iteration_df = iteration_df.merge(
        po_df[['team_id', 'playoff_place']], 
        on='team_id', 
        how='left'
        )
    iteration_df = iteration_df.merge(
        sc_df[['team_id', 'supercup_place']], 
        on='team_id', 
        how='left'
        )
    iteration_df['is_classico'] = classico_flag
    iteration_df['iteration_id'] = itr

    res_df = pd.concat([res_df, iteration_df])

print("Probability of classico in supercup final: ", sum(classico_flas_lst) / n_iterations)

Probability of classico in supercup final:  0.44


In [16]:
res_df['is_liga_1'] = np.where(res_df['roundrobin_place'] == 1, 1, 0)
res_df['is_liga_2'] = np.where(res_df['roundrobin_place'] == 2, 1, 0)
res_df['is_liga_3'] = np.where(res_df['roundrobin_place'] == 3, 1, 0)
res_df['is_cup_winner'] = np.where(res_df['playoff_place'] == 1, 1, 0)
res_df['is_cup_finalist'] = np.where(res_df['playoff_place'] == 2, 1, 0)
res_df['is_supertcup_1'] = np.where(res_df['supercup_place'] == 1, 1, 0)
res_df['is_supertcup_2'] = np.where(res_df['supercup_place'] == 2, 1, 0)
res_df['is_supertcup_3'] = np.where(res_df['supercup_place'] == 3, 1, 0)
res_df['is_supertcup_4'] = np.where(res_df['supercup_place'] == 4, 1, 0)


team_stat_df = res_df.groupby('team_name').agg(
    iterations = ('iteration_id', 'nunique'),
    liga_1 = ('is_liga_1', 'sum'),
    liga_2 = ('is_liga_2', 'sum'),
    liga_3 = ('is_liga_3', 'sum'),
    cup_winner = ('is_cup_winner', 'sum'),
    cup_finalist = ('is_cup_finalist', 'sum'),
    supertcup_1 = ('is_supertcup_1', 'sum'),
    supertcup_2 = ('is_supertcup_2', 'sum'),
    supertcup_3 = ('is_supertcup_3', 'sum'),
    supertcup_4 = ('is_supertcup_4', 'sum'),
    classico_in_supercup_final = ('is_classico', 'sum'),
).reset_index().sort_values('liga_1', ascending=False)

In [18]:
team_stat_df

,team_name,iterations,liga_1,liga_2,liga_3,cup_winner,cup_finalist,supertcup_1,supertcup_2,supertcup_3,supertcup_4,classico_in_supercup_final
3,Barcelona,100,47,35,16,29,14,30,31,20,14,44
26,Real Madrid,100,33,39,27,21,16,39,30,16,8,44
2,Atletico,100,20,23,47,14,13,17,21,24,21,44
1,Almeria,100,0,0,0,0,0,0,0,0,0,44
4,Betis,100,0,0,2,5,11,6,2,6,10,44
5,Bilbao,100,0,0,0,4,7,1,1,7,6,44
6,Burgos,100,0,0,0,0,1,0,0,0,1,44
0,Alaves,100,0,0,0,0,1,0,0,0,1,44
8,Castellon,100,0,0,0,0,0,0,0,0,0,44
9,Celta,100,0,0,0,6,7,0,2,5,7,44


In [ ]:
classico_flas_lst = []
for itr in range(n):
    # playof - fast and popular way, bit hig dispersional
    # log with game results after simulation
    # half seed - seed by start rating, best 2 teams can meet only in final
    log_df = t.simulate_playoff_tourn(teams_dct, seed_mode='half')
    # usefull table to look at different parameters
    team_df = t.team_table(log_df, teams_dct)
    # tournament table
    po_df = t.make_res_table(log_df, teams_dct)

    # round robin - the best for dispersion, but long schedule
    # two rounds by default
    schedule = t.get_round_robin_schedule(teams_dct)
    # log with game results after simulation
    log_df = t.simulate_matches_by_schedule(teams_dct, schedule, total=(0,1), sigma=(2.5, 2.6), last_match_id=0)
    # tournament table
    liga_df = t.make_res_table(log_df, teams_dct)

    # сначала финалист
    po_df = po_df.sort_values(by='playoff_place', ascending=True)
    sc_df_teams_lst = list(po_df['team_id'].astype(int).values[0:1])
    # добираем из чемпионата пока число участников не станет 4
    liga_df = liga_df.sort_values(by='roundrobin_place', ascending=True)
    i = 0
    while len(sc_df_teams_lst) < 2:
        cand = liga_df['team_id'].iloc[i]
        if cand not in sc_df_teams_lst:
            sc_df_teams_lst.append(cand)
        i += 1

    sc_dct = {}
    for i in range(len(sc_df_teams_lst)):
        sc_dct[sc_df_teams_lst[i]] = teams_dct[sc_df_teams_lst[i]]



    log_df = t.simulate_playoff_tourn(sc_dct, seed_mode='half')
    # usefull table to look at different parameters
    team_df = t.team_table(log_df, sc_dct)
    # tournament table
    sc_df = t.make_res_table(log_df, sc_dct)

    sc_final_teams = sc_df[sc_df['playoff_place'] < 3]['team_id'].values
    sc_final_teams.sort()

    classico_flag = 0
    if (2 in sc_final_teams) and (1 in sc_final_teams):
        classico_flag = 1
    else:
        classico_flag = 0

    classico_flas_lst.append(classico_flag)


In [ ]:
sum(classico_flas_lst) / n